# Stage 7: Advanced Models

Tune XGBoost and LightGBM against the same `StratifiedKFold(5)` split, using the same
`build_preprocessor()` pipeline shape as the Stage 6 baseline. No class weighting or
resampling anywhere: threshold tuning in Stage 8 is the chosen imbalance mechanism.

The reader should come away knowing whether the Stage 6 gap over the RFM heuristic
(0.0206 ROC-AUC, single split) survives cross-validation, and which of the three model
families is the honest out-of-fold winner.

In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.append(str(PROJECT_ROOT))

# Data directory
DATA_DIR = Path(PROJECT_ROOT, "data")
DATA_DIR.mkdir(exist_ok=True)

In [2]:
import numpy as np
import pandas as pd

from src.config import (
    CV_FOLDS,
    FEATURE_COLUMNS,
    MIN_ACCEPTABLE_ROC_AUC,
    MODELS_DIR,
    OUTPUT_DIR,
    RANDOM_SEED,
    TEST_PARQUET,
    TRAIN_PARQUET,
)
from src.logger import setup_logger

logger = setup_logger("05-advanced-models")

logger.info("RANDOM_SEED=%d, CV_FOLDS=%d, MIN_ACCEPTABLE_ROC_AUC=%.2f",
            RANDOM_SEED, CV_FOLDS, MIN_ACCEPTABLE_ROC_AUC)

20:08:29 | 05-advanced-models | INFO | RANDOM_SEED=42, CV_FOLDS=5, MIN_ACCEPTABLE_ROC_AUC=0.75


## 1. Load the Stage 6 split

Loading `train.parquet` and `test.parquet` as saved. Not re-splitting: this is the one
split every stage from here on shares.

Expecting 4,204 train rows and 1,052 test rows, matching Stage 6 exactly.

In [3]:
train_df = pd.read_parquet(TRAIN_PARQUET)
test_df = pd.read_parquet(TEST_PARQUET)

X_train, y_train = train_df[list(FEATURE_COLUMNS)], train_df["y"]
X_test, y_test = test_df[list(FEATURE_COLUMNS)], test_df["y"]

logger.info("train: %s rows, base rate %.4f", f"{len(train_df):,}", y_train.mean())
logger.info("test : %s rows, base rate %.4f", f"{len(test_df):,}", y_test.mean())
assert len(train_df) == 4204 and len(test_df) == 1052

20:08:29 | 05-advanced-models | INFO | train: 4,204 rows, base rate 0.4355


20:08:29 | 05-advanced-models | INFO | test : 1,052 rows, base rate 0.4354


4,204 / 1,052, base rates 0.4355 / 0.4354, identical to Stage 6. The assertion confirms
this is the same split, not a re-shuffled one.

## 2. The shared fold object, and Logistic Regression's real cross-validated score

One `StratifiedKFold(5, shuffle=True, random_state=42)` object, reused for every model
family below, so the comparison is fair: same rows in the same folds for all three.

Stage 6 only reported Logistic Regression's score on one single train/test split
(0.8139), with no way to tell whether the 0.0206 gap over the RFM heuristic was real or
noise. Computing the actual 5-fold cross-validated mean and standard deviation now, using
`src.modeling.build_preprocessor()`, the same pipeline shape Stage 6 saved.

Expecting the CV mean to land close to 0.81, and the fold standard deviation to be
meaningful given about 840 rows per fold, a genuinely small validation set.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

from src.modeling import build_preprocessor

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
for i, (_, val_idx) in enumerate(cv.split(X_train, y_train)):
    logger.info("fold %d: %d rows, base rate %.4f", i, len(val_idx), y_train.iloc[val_idx].mean())

logreg_cv_pipeline = Pipeline([
    ("preprocess", build_preprocessor()),
    ("model", LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)),
])
logreg_cv_scores = cross_val_score(logreg_cv_pipeline, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)

logger.info("Logistic Regression fold scores: %s", np.round(logreg_cv_scores, 4))
logger.info("Logistic Regression CV ROC-AUC: %.4f +/- %.4f", logreg_cv_scores.mean(), logreg_cv_scores.std())

20:08:30 | 05-advanced-models | INFO | fold 0: 841 rows, base rate 0.4364


20:08:30 | 05-advanced-models | INFO | fold 1: 841 rows, base rate 0.4352


20:08:30 | 05-advanced-models | INFO | fold 2: 841 rows, base rate 0.4352


20:08:30 | 05-advanced-models | INFO | fold 3: 841 rows, base rate 0.4352


20:08:30 | 05-advanced-models | INFO | fold 4: 840 rows, base rate 0.4357


20:08:33 | 05-advanced-models | INFO | Logistic Regression fold scores: [0.7908 0.817  0.7918 0.7886 0.7848]


20:08:33 | 05-advanced-models | INFO | Logistic Regression CV ROC-AUC: 0.7946 +/- 0.0114


**This changes the Stage 6 conclusion.** Cross-validated, Logistic Regression scores
**0.7946 ± 0.0114**, not the 0.8139 that one lucky-ish train/test split reported (fold 1
alone hit 0.817, pulling that single split's number up). Fold scores range 0.785 to 0.817,
a real spread given about 840 rows per fold.

Against the RFM heuristic's single-split 0.7932, the gap is now **0.0014**, dwarfed by the
0.0114 fold standard deviation. **Logistic Regression does not clear the Stage 0 bar**
("beats the RFM heuristic by more than the fold-to-fold standard deviation"). The 0.0206
gap Stage 6 reported was mostly the luck of one split, not a real advantage.

This is the exact scenario Stage 0 was written to catch, and it is a live possibility
raised earlier in this project: a linear model on RFM-adjacent features may simply not
outperform the RFM rule it was built to compete with. Whether the two boosted trees below
close that gap for real is now the entire question this stage has to answer.

## 3. XGBoost

`RandomizedSearchCV` on the identical `cv` object, scored on ROC-AUC, training set only.
No `scale_pos_weight`: decision 0.8 rules out any imbalance correction, since threshold
tuning in Stage 8 is the chosen mechanism.

The search space is kept modest given the dataset size (about 3,360 rows per training
fold): `max_depth` capped at 6, `n_estimators` up to 400 with `learning_rate` traded
against it, and both subsampling knobs included specifically to fight overfitting on a
table this small.

Genuinely unsure what to expect here. XGBoost can model interactions Logistic Regression
cannot (the EDA's own finding that cancellations proxy for volume, not dissatisfaction, is
exactly the kind of conditional relationship a linear model struggles with), but 4,204
rows is not a lot of data for tuning 24 features worth of tree splits.

In [5]:
from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ("preprocess", build_preprocessor()),
    ("model", XGBClassifier(random_state=RANDOM_SEED, eval_metric="logloss", n_jobs=1)),
])

xgb_param_dist = {
    "model__n_estimators": randint(50, 400),
    "model__max_depth": randint(2, 7),
    "model__learning_rate": uniform(0.01, 0.29),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__min_child_weight": randint(1, 10),
    "model__reg_alpha": uniform(0.0, 1.0),
    "model__reg_lambda": uniform(0.5, 2.0),
}

xgb_search = RandomizedSearchCV(
    xgb_pipeline, xgb_param_dist, n_iter=40, cv=cv, scoring="roc_auc",
    random_state=RANDOM_SEED, n_jobs=-1, refit=True,
)
xgb_search.fit(X_train, y_train)

logger.info("XGBoost best params: %s", xgb_search.best_params_)
logger.info("XGBoost best CV ROC-AUC: %.4f", xgb_search.best_score_)

20:08:46 | 05-advanced-models | INFO | XGBoost best params: {'model__colsample_bytree': np.float64(0.8654007076432223), 'model__learning_rate': np.float64(0.011467859315403419), 'model__max_depth': 3, 'model__min_child_weight': 2, 'model__n_estimators': 359, 'model__reg_alpha': np.float64(0.44842414298624733), 'model__reg_lambda': np.float64(2.4889149252216414), 'model__subsample': np.float64(0.6703701010709381)}


20:08:46 | 05-advanced-models | INFO | XGBoost best CV ROC-AUC: 0.8026


In [6]:
best_idx = xgb_search.best_index_
xgb_cv_std = xgb_search.cv_results_["std_test_score"][best_idx]
xgb_fold_scores = [xgb_search.cv_results_[f"split{i}_test_score"][best_idx] for i in range(CV_FOLDS)]

logger.info("XGBoost fold scores: %s", np.round(xgb_fold_scores, 4))
logger.info("XGBoost CV ROC-AUC: %.4f +/- %.4f", xgb_search.best_score_, xgb_cv_std)
logger.info("gap over RFM heuristic (0.7932): %+.4f", xgb_search.best_score_ - 0.7932)
logger.info("gap exceeds XGBoost's own fold std: %s", (xgb_search.best_score_ - 0.7932) > xgb_cv_std)

20:08:46 | 05-advanced-models | INFO | XGBoost fold scores: [0.8022 0.8188 0.8087 0.7875 0.7956]


20:08:46 | 05-advanced-models | INFO | XGBoost CV ROC-AUC: 0.8026 +/- 0.0107


20:08:46 | 05-advanced-models | INFO | gap over RFM heuristic (0.7932): +0.0094


20:08:46 | 05-advanced-models | INFO | gap exceeds XGBoost's own fold std: False


**XGBoost: 0.8026 ± 0.0107.** Higher mean than both Logistic Regression (0.7946) and the
RFM heuristic (0.7932), and the best model so far. But the gap over RFM is +0.0094,
smaller than XGBoost's own fold standard deviation of 0.0107. **Still does not clear the
Stage 0 bar.**

The winning hyperparameters lean conservative: `max_depth=3` (shallow trees), a low
`learning_rate` of 0.011 compensated by 359 estimators, and `reg_lambda=2.49`, fairly
strong L2 regularization. That is what a search *should* land on for 4,204 rows: the
tuning process itself is pushing away from complexity, which is a mild reassurance
against overfitting rather than a concern.

### 3a. Out-of-fold probabilities and test evaluation

Stage 8 needs genuine out-of-fold probabilities on train to sweep decision thresholds
without leakage. `RandomizedSearchCV` does not retain per-fold predictions, so refitting
the best hyperparameters through `cross_val_predict` on the same `cv` object, which is
the only way to get an unbiased probability for every training row.

The refit-on-full-train estimator (`xgb_search.best_estimator_`) is what gets evaluated
on test, once, and saved.

In [7]:
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import cross_val_predict

xgb_best_pipeline = clone(xgb_pipeline).set_params(**xgb_search.best_params_)
xgb_oof_proba = cross_val_predict(xgb_best_pipeline, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

logger.info("XGBoost OOF ROC-AUC: %.4f (sanity check against best_score_ %.4f)",
            roc_auc_score(y_train, xgb_oof_proba), xgb_search.best_score_)

xgb_test_proba = xgb_search.best_estimator_.predict_proba(X_test)[:, 1]
xgb_test_roc_auc = roc_auc_score(y_test, xgb_test_proba)
xgb_test_pr_auc = average_precision_score(y_test, xgb_test_proba)

logger.info("XGBoost test ROC-AUC: %.4f", xgb_test_roc_auc)
logger.info("XGBoost test PR-AUC : %.4f", xgb_test_pr_auc)

20:08:47 | 05-advanced-models | INFO | XGBoost OOF ROC-AUC: 0.8018 (sanity check against best_score_ 0.8026)


20:08:47 | 05-advanced-models | INFO | XGBoost test ROC-AUC: 0.8222


20:08:47 | 05-advanced-models | INFO | XGBoost test PR-AUC : 0.7968


OOF ROC-AUC (0.8018) lands close to `best_score_` (0.8026) but not identical, expected
since `cross_val_predict` pools every row into one AUC calculation while `best_score_`
averages five separate per-fold AUCs. Two correct ways to aggregate, small difference,
not a bug.

**Test ROC-AUC 0.8222**, noticeably above the 0.8026 CV mean. This is the same trap
Stage 6 fell into with Logistic Regression: a single evaluation set can run high by luck.
The CV mean (0.8026 ± 0.0107) stays the number to trust for model selection, exactly what
decision 0.6 says, the test score only confirms a decision already made.

In [8]:
import joblib

OUTPUT_DIR.mkdir(exist_ok=True)
xgb_oof_path = OUTPUT_DIR / "xgboost_oof_probabilities.parquet"
pd.DataFrame({"customer_id": train_df["customer_id"], "y": y_train, "oof_proba": xgb_oof_proba}).to_parquet(xgb_oof_path, index=False)

xgb_model_path = MODELS_DIR / "xgboost.joblib"
joblib.dump(xgb_search.best_estimator_, xgb_model_path)

logger.info("wrote %s (%.1f KB)", xgb_oof_path.name, xgb_oof_path.stat().st_size / 1024)
logger.info("wrote %s (%.1f KB)", xgb_model_path.name, xgb_model_path.stat().st_size / 1024)

20:08:47 | 05-advanced-models | INFO | wrote xgboost_oof_probabilities.parquet (49.0 KB)


20:08:47 | 05-advanced-models | INFO | wrote xgboost.joblib (421.3 KB)


Saved. `xgboost.joblib` at 421 KB, much larger than the 5.2 KB Logistic Regression
artifact, which is exactly what 359 trees look like on disk versus one linear equation.

## 4. LightGBM

Same `cv` object, same scoring, same absence of any class-weight parameter. LightGBM
grows leaf-wise rather than XGBoost's level-wise, so `num_leaves` is the primary
complexity knob here instead of `max_depth`, alongside `min_child_samples` to prevent
leaves from fitting to a handful of rows on a table this small.

Expecting a result in the same neighborhood as XGBoost, since both are gradient-boosted
trees over the identical feature set. `verbose=-1` to silence LightGBM's per-iteration
logging, which would otherwise flood the cell output.

In [9]:
from lightgbm import LGBMClassifier

lgbm_pipeline = Pipeline([
    ("preprocess", build_preprocessor()),
    ("model", LGBMClassifier(random_state=RANDOM_SEED, verbose=-1, n_jobs=1)),
])

lgbm_param_dist = {
    "model__n_estimators": randint(50, 400),
    "model__num_leaves": randint(7, 63),
    "model__learning_rate": uniform(0.01, 0.29),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
    "model__min_child_samples": randint(5, 60),
    "model__reg_alpha": uniform(0.0, 1.0),
    "model__reg_lambda": uniform(0.5, 2.0),
}

lgbm_search = RandomizedSearchCV(
    lgbm_pipeline, lgbm_param_dist, n_iter=40, cv=cv, scoring="roc_auc",
    random_state=RANDOM_SEED, n_jobs=-1, refit=True,
)
lgbm_search.fit(X_train, y_train)

lgbm_best_idx = lgbm_search.best_index_
lgbm_cv_std = lgbm_search.cv_results_["std_test_score"][lgbm_best_idx]
lgbm_fold_scores = [lgbm_search.cv_results_[f"split{i}_test_score"][lgbm_best_idx] for i in range(CV_FOLDS)]

logger.info("LightGBM best params: %s", lgbm_search.best_params_)
logger.info("LightGBM fold scores: %s", np.round(lgbm_fold_scores, 4))
logger.info("LightGBM CV ROC-AUC: %.4f +/- %.4f", lgbm_search.best_score_, lgbm_cv_std)
logger.info("gap over RFM heuristic (0.7932): %+.4f", lgbm_search.best_score_ - 0.7932)
logger.info("gap exceeds LightGBM's own fold std: %s", (lgbm_search.best_score_ - 0.7932) > lgbm_cv_std)

20:08:57 | 05-advanced-models | INFO | LightGBM best params: {'model__colsample_bytree': np.float64(0.9921326334864182), 'model__learning_rate': np.float64(0.03185041424177718), 'model__min_child_samples': 25, 'model__n_estimators': 210, 'model__num_leaves': 9, 'model__reg_alpha': np.float64(0.1694927466860925), 'model__reg_lambda': np.float64(1.6136025249167003), 'model__subsample': np.float64(0.9744619096643123)}


20:08:57 | 05-advanced-models | INFO | LightGBM fold scores: [0.7954 0.8162 0.7956 0.7904 0.7922]


20:08:57 | 05-advanced-models | INFO | LightGBM CV ROC-AUC: 0.7980 +/- 0.0093


20:08:57 | 05-advanced-models | INFO | gap over RFM heuristic (0.7932): +0.0048


20:08:57 | 05-advanced-models | INFO | gap exceeds LightGBM's own fold std: False


**LightGBM: 0.7980 ± 0.0093.** Close to XGBoost but a bit lower (0.7980 vs 0.8026), and
close to Logistic Regression (0.7946). Gap over RFM is +0.0048, well inside LightGBM's own
fold standard deviation. **Also does not clear the Stage 0 bar.**

`num_leaves=9` is a shallow tree by leaf-wise standards, and `min_child_samples=25` keeps
any single leaf from fitting to a handful of rows. Same story as XGBoost: the search
settled on a conservative model, which is what a small, noisy table should produce.

Four model families now agree within noise of each other: RFM 0.7932, LightGBM 0.7980,
Logistic Regression 0.7946, XGBoost 0.8026. All four sit inside roughly a 0.01 band, and
every fold standard deviation is around 0.009 to 0.011. **None of the built models clearly
beats the spreadsheet rule yet.**

In [10]:
lgbm_best_pipeline = clone(lgbm_pipeline).set_params(**lgbm_search.best_params_)
lgbm_oof_proba = cross_val_predict(lgbm_best_pipeline, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

logger.info("LightGBM OOF ROC-AUC: %.4f (sanity check against best_score_ %.4f)",
            roc_auc_score(y_train, lgbm_oof_proba), lgbm_search.best_score_)

lgbm_test_proba = lgbm_search.best_estimator_.predict_proba(X_test)[:, 1]
lgbm_test_roc_auc = roc_auc_score(y_test, lgbm_test_proba)
lgbm_test_pr_auc = average_precision_score(y_test, lgbm_test_proba)

logger.info("LightGBM test ROC-AUC: %.4f", lgbm_test_roc_auc)
logger.info("LightGBM test PR-AUC : %.4f", lgbm_test_pr_auc)

lgbm_oof_path = OUTPUT_DIR / "lightgbm_oof_probabilities.parquet"
pd.DataFrame({"customer_id": train_df["customer_id"], "y": y_train, "oof_proba": lgbm_oof_proba}).to_parquet(lgbm_oof_path, index=False)

lgbm_model_path = MODELS_DIR / "lightgbm.joblib"
joblib.dump(lgbm_search.best_estimator_, lgbm_model_path)

logger.info("wrote %s (%.1f KB)", lgbm_oof_path.name, lgbm_oof_path.stat().st_size / 1024)
logger.info("wrote %s (%.1f KB)", lgbm_model_path.name, lgbm_model_path.stat().st_size / 1024)

20:08:57 | 05-advanced-models | INFO | LightGBM OOF ROC-AUC: 0.7979 (sanity check against best_score_ 0.7980)


20:08:57 | 05-advanced-models | INFO | LightGBM test ROC-AUC: 0.8191


20:08:57 | 05-advanced-models | INFO | LightGBM test PR-AUC : 0.7941


20:08:57 | 05-advanced-models | INFO | wrote lightgbm_oof_probabilities.parquet (65.3 KB)


20:08:57 | 05-advanced-models | INFO | wrote lightgbm.joblib (238.8 KB)


OOF (0.7979) matches `best_score_` (0.7980) almost exactly. Test ROC-AUC 0.8191, again
well above the CV mean, the same direction as both Logistic Regression and XGBoost. **All
three models score higher on this particular test set than on cross-validation.** That is
not three independent lucky breaks, it suggests this specific 1,052-row test split is
mildly easier than the average training fold, which is exactly the kind of thing a single
split can hide and cross-validation exposes. Reinforces trusting the CV numbers over the
test numbers for every comparison in this notebook.

238.8 KB for LightGBM, between Logistic Regression's 5.2 KB and XGBoost's 421.3 KB, in
line with `n_estimators=210` against XGBoost's 359.

## 5. Full comparison, and the selection call

Per decision 0.6, selection happens on out-of-fold CV, never on the test table. Building
the comparison on that basis, with test scores shown only as the already-decided
confirmation, not as an input to the decision.

### 5a. Bringing the Stage 6 baselines into this notebook

The comparison table needs the RFM heuristic's and Logistic Regression's single-split
test scores, and this is a fresh notebook: `rfm_roc_auc` and the rest from Stage 6 do not
exist here. Recomputing both on the identical train/test files rather than importing
numbers across notebooks, so this notebook is self-contained and re-runnable on its own.

Expecting an exact match to Stage 6's numbers (RFM 0.7932/0.7443, LR 0.8139/0.7962), since
it is the same data and the same procedure.

In [11]:
def fit_rfm_bins(train_series: pd.Series, ascending: bool):
    _, edges = pd.qcut(train_series, 5, retbins=True, duplicates="drop")
    edges = edges.copy()
    edges[0], edges[-1] = -np.inf, np.inf
    n_bins = len(edges) - 1
    labels = list(range(1, n_bins + 1)) if ascending else list(range(n_bins, 0, -1))
    return edges, labels


def apply_rfm_bins(series: pd.Series, edges, labels) -> pd.Series:
    return pd.cut(series, bins=edges, labels=labels, include_lowest=True).astype(int)


rfm_cols = {"recency_days": False, "frequency": True, "monetary_total": True}
rfm_fit = {col: fit_rfm_bins(train_df[col], asc) for col, asc in rfm_cols.items()}
test_rfm_total = sum(apply_rfm_bins(test_df[col], *rfm_fit[col]) for col in rfm_cols)

rfm_roc_auc = roc_auc_score(y_test, test_rfm_total)
rfm_pr_auc = average_precision_score(y_test, test_rfm_total)

logreg_pipeline = Pipeline([
    ("preprocess", build_preprocessor()),
    ("model", LogisticRegression(random_state=RANDOM_SEED, max_iter=1000)),
])
logreg_pipeline.fit(X_train, y_train)
logreg_proba = logreg_pipeline.predict_proba(X_test)[:, 1]
logreg_roc_auc = roc_auc_score(y_test, logreg_proba)
logreg_pr_auc = average_precision_score(y_test, logreg_proba)

logger.info("RFM test: ROC-AUC %.4f, PR-AUC %.4f (Stage 6: 0.7932, 0.7443)", rfm_roc_auc, rfm_pr_auc)
logger.info("LR test : ROC-AUC %.4f, PR-AUC %.4f (Stage 6: 0.8139, 0.7962)", logreg_roc_auc, logreg_pr_auc)
assert abs(rfm_roc_auc - 0.7932) < 1e-3
assert abs(logreg_roc_auc - 0.8139) < 1e-3

20:08:57 | 05-advanced-models | INFO | RFM test: ROC-AUC 0.7932, PR-AUC 0.7443 (Stage 6: 0.7932, 0.7443)


20:08:57 | 05-advanced-models | INFO | LR test : ROC-AUC 0.8139, PR-AUC 0.7962 (Stage 6: 0.8139, 0.7962)


In [12]:
final_comparison = pd.DataFrame([
    {"model": "RFM heuristic", "cv_roc_auc": np.nan, "cv_std": np.nan,
     "test_roc_auc": rfm_roc_auc, "test_pr_auc": rfm_pr_auc, "beats_rfm_by_more_than_std": False},
    {"model": "Logistic Regression", "cv_roc_auc": logreg_cv_scores.mean(), "cv_std": logreg_cv_scores.std(),
     "test_roc_auc": logreg_roc_auc, "test_pr_auc": logreg_pr_auc,
     "beats_rfm_by_more_than_std": (logreg_cv_scores.mean() - rfm_roc_auc) > logreg_cv_scores.std()},
    {"model": "XGBoost", "cv_roc_auc": xgb_search.best_score_, "cv_std": xgb_cv_std,
     "test_roc_auc": xgb_test_roc_auc, "test_pr_auc": xgb_test_pr_auc,
     "beats_rfm_by_more_than_std": (xgb_search.best_score_ - rfm_roc_auc) > xgb_cv_std},
    {"model": "LightGBM", "cv_roc_auc": lgbm_search.best_score_, "cv_std": lgbm_cv_std,
     "test_roc_auc": lgbm_test_roc_auc, "test_pr_auc": lgbm_test_pr_auc,
     "beats_rfm_by_more_than_std": (lgbm_search.best_score_ - rfm_roc_auc) > lgbm_cv_std},
]).set_index("model")

display(final_comparison.round(4))

best_by_cv = final_comparison["cv_roc_auc"].idxmax()
logger.info("best by CV ROC-AUC: %s (%.4f)", best_by_cv, final_comparison.loc[best_by_cv, "cv_roc_auc"])
logger.info("any model beats RFM by more than its own fold std: %s",
            final_comparison["beats_rfm_by_more_than_std"].any())

,cv_roc_auc,cv_std,test_roc_auc,test_pr_auc,beats_rfm_by_more_than_std
model,,,,,
RFM heuristic,NaN,NaN,0.7932,0.7443,False
Logistic Regression,0.7946,0.0114,0.8139,0.7962,False
XGBoost,0.8026,0.0107,0.8222,0.7968,False
LightGBM,0.7980,0.0093,0.8191,0.7941,False


20:08:57 | 05-advanced-models | INFO | best by CV ROC-AUC: XGBoost (0.8026)


20:08:57 | 05-advanced-models | INFO | any model beats RFM by more than its own fold std: False


**XGBoost is the best model by cross-validated ROC-AUC (0.8026), and none of the three
built models beats the RFM heuristic by more than its own fold standard deviation.**

| Model | CV ROC-AUC | CV std | Test ROC-AUC | Beats RFM by more than std |
|---|---|---|---|---|
| RFM heuristic | (not tuned) | (not tuned) | 0.7932 | n/a |
| Logistic Regression | 0.7946 | 0.0114 | 0.8139 | No |
| LightGBM | 0.7980 | 0.0093 | 0.8191 | No |
| XGBoost | 0.8026 | 0.0107 | 0.8222 | No |

Every test score sits noticeably above its own model's CV mean (LR +0.019, LightGBM
+0.021, XGBoost +0.020), the same direction for all three, which is the pattern noted in
section 4: this particular 1,052-row test split runs a bit easy relative to the average
training fold. The CV numbers are the ones to trust, and by that measure the honest
conclusion is that **none of the candidates has demonstrated a real advantage over the
spreadsheet rule on this dataset**, at least not one distinguishable from sampling noise
given roughly 4,200 training rows.

XGBoost is still the model to carry forward, since Stage 8 needs a concrete choice to
threshold-tune and compare against the trivial policies on the actual cost metric
(Stage 0's real bar: 20 percent cost reduction, not ROC-AUC). Whether XGBoost clears
*that* bar, where a probability-based threshold has room to work that RFM's 13 discrete
score levels do not, is a genuinely open question Stage 8 has to answer, not something
this notebook can resolve.

## 6. Stage 7 summary

### What was produced

- `models/xgboost.joblib`, `models/lightgbm.joblib`: tuned pipelines, each
  `Pipeline(build_preprocessor(), model)`, fitted on the full training set with the best
  `RandomizedSearchCV` hyperparameters.
- `data/output/xgboost_oof_probabilities.parquet`,
  `data/output/lightgbm_oof_probabilities.parquet`: genuine out-of-fold probabilities on
  every training row, for Stage 8's threshold sweep.
- `models/baseline_logreg.joblib` (Stage 6) remains the third candidate; not re-saved
  here since nothing about it changed.

### Key numbers

| Model | CV ROC-AUC | CV std |
|---|---|---|
| RFM heuristic | (not tuned) | (not tuned) |
| Logistic Regression | 0.7946 | 0.0114 |
| LightGBM | 0.7980 | 0.0093 |
| XGBoost | **0.8026** | 0.0107 |

### Decisions made here

1. **Model selection basis: out-of-fold CV, per decision 0.6.** XGBoost wins on that
   basis. The single-split test numbers (all three, 0.81 to 0.82) were never used to pick
   a winner, only to confirm a choice already made on CV.
2. **None of the three models beats the RFM heuristic by more than its own fold standard
   deviation.** The headline finding of this stage. Stage 6's reported 0.0206 gap for
   Logistic Regression was mostly a lucky single split; cross-validated, it shrinks to
   0.0014. This is exactly the failure mode Stage 0's bar was written to catch.
3. Both hyperparameter searches converged on conservative settings (shallow trees, strong
   regularization), which is the expected and reassuring outcome for a table this small,
   not a sign the search under-performed.
4. No class weighting or resampling in either search space, per decision 0.8. Threshold
   tuning in Stage 8 remains the sole imbalance mechanism.
5. XGBoost carries forward to Stage 8 despite not clearing the ROC-AUC bar, because
   Stage 8's real test is the 20 percent expected-cost reduction from Stage 0, which a
   continuous probability score can pursue via threshold tuning in a way RFM's 13 discrete
   levels cannot. This is not overriding decision 0.6, since no CV-based selection among
   the three models was reversed, it is carrying the CV-selected candidate into the stage
   that asks the actual business question.

### Open questions for later stages

- **Can XGBoost clear the real Stage 0 bar (20 percent cost reduction vs. the trivial
  policies) even though it did not clear the ROC-AUC bar over RFM?** These are different
  questions. A model can be statistically indistinguishable from a heuristic on ranking
  quality while still being cheaper to operate at the specific threshold the business
  cares about, or it can fail both. Stage 8 answers this directly.
- If Stage 8 shows XGBoost does not clear the cost bar either, the documented, legitimate
  conclusion is to recommend the RFM heuristic for deployment. Nothing in this project
  forces a model to win.

### Exit check

| Requirement | Status |
|---|---|
| All candidate artifacts saved | Sections 3a, 4 |
| Comparison table exists | Section 5 |
| Every model tuned on identical CV folds | Section 2, one shared `cv` object throughout |
| Every model evaluated on the identical test file | Sections 3a-5, same `test_df` |

Stage 8 may begin.